# CS781 Capstone — Phase 2: Legal NER (L-NER)

Name: Aditya Kumar | Roll No: 240059 | Email: adityaj24@iitk.ac.in

In [1]:
import os, re, math, json, time, random, zipfile, csv, copy, glob, shutil, subprocess
from bisect import bisect_left, bisect_right
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

cuda 2


In [2]:
   !find /kaggle/input -maxdepth 4 | head -30

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sparkles1234
/kaggle/input/datasets/sparkles1234/hybrid-240059
/kaggle/input/datasets/sparkles1234/hybrid-240059/heldout_truth.json
/kaggle/input/datasets/sparkles1234/hybrid-240059/hybrid_model.pt
/kaggle/input/datasets/sparkles1234/hybrid-240059/heldout_prediction.json
/kaggle/input/datasets/sparkles1234/hybrid-240059/prediction.json
/kaggle/input/datasets/sparkles1234/hybrid-240059/experiment_log.csv
/kaggle/input/datasets/sparkles1234/hybrid-240059/run_info.json
/kaggle/input/datasets/sparkles1234/hybrid-240059/submission
/kaggle/input/datasets/sparkles1234/hybrid-240059/loss.png
/kaggle/input/datasets/sparkles1234/hybrid-240059/bpe_tokenizer.json
/kaggle/input/datasets/adityakumar344
/kaggle/input/datasets/adityakumar344/evaluate-phase2-script
/kaggle/input/datasets/adityakumar344/evaluate-phase2-script/evaluate_phase2.py


In [ ]:
ARCH = "rope_qknorm"          
BIDIRECTIONAL = True
SMOKE = True            

RUNS = [
    dict(method="full"),
    dict(method="lorma"),
    dict(method="full", kd=True),
    dict(method="lora"),
    dict(method="adapter"),
    dict(method="bitfit"),
    dict(method="partial"),
]
TIME_BUDGET_H = 9.0
STUDENT_KEEP_EVERY = 2
KD_T, KD_ALPHA = 2.0, 0.5

hf_token = os.environ.get("HF_TOKEN")
HF_REPO, P2_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_2_LNER"
LOCAL_DATA_DIR = None
HF_BACKUP_REPO = None

def find_one(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input - attach the dataset that contains it")
    return hits[0]

_model_hits = sorted(glob.glob(f"/kaggle/input/**/{ARCH}_model.pt", recursive=True))
MODEL_PATH = _model_hits[0] if _model_hits else None
if MODEL_PATH is None:
    print("no Phase-1 model found - fine for ensemble-only sessions, not for training")
_tok_near = os.path.join(os.path.dirname(MODEL_PATH), "bpe_tokenizer.json") if MODEL_PATH else None
TOKENIZER_PATH = _tok_near if _tok_near and os.path.exists(_tok_near) else find_one("bpe_tokenizer.json")
_ev = sorted(glob.glob("/kaggle/input/**/evaluate_phase2*.py", recursive=True))
EVAL_SCRIPT = _ev[0] if _ev else None
print("model:", MODEL_PATH, "| tokenizer:", TOKENIZER_PATH, "| eval script:", EVAL_SCRIPT)

DEV_MODE, DEV_FRAC = "random", 0.12
WINDOW, TRAIN_STRIDE, INFER_STRIDE = 512, 384, 256
KEEP_EMPTY = 0.3
O_WEIGHT = 0.5
DECODE = "viterbi"

EPOCHS, BATCH_SIZE, GRAD_ACCUM = (1 if SMOKE else 4), 16, 1
DEFAULT_LR = {"full": 1e-4, "partial": 2e-4, "lora": 5e-4, "lorma": 5e-4, "adapter": 5e-4, "bitfit": 1e-3}
HEAD_LR_MULT, WEIGHT_DECAY, WARMUP_FRAC = 5.0, 0.01, 0.06
PEFT = dict(r=16, alpha=32, targets=("q_proj", "v_proj"), bottleneck=64, unfreeze_last=2)
SAVE_LP = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK_ROOT = f"/kaggle/working/cs781_phase2{'_smoke' if SMOKE else ''}"
SUB_DIR = "/kaggle/working/submissions" + ("_smoke" if SMOKE else "")
os.makedirs(WORK_ROOT, exist_ok=True); os.makedirs(SUB_DIR, exist_ok=True)

model: /kaggle/input/datasets/sparkles1234/hybrid-240059/hybrid_model.pt | tokenizer: /kaggle/input/datasets/sparkles1234/hybrid-240059/bpe_tokenizer.json | eval script: /kaggle/input/datasets/adityakumar344/evaluate-phase2-script/evaluate_phase2.py


In [4]:
class BPE:
    """Byte-level BPE tokenizer."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL):
            self.special_to_id[t] = 256 + i
            self.id_to_bytes[256 + i] = t.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok

    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            mergeable = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not mergeable:
                break
            pair = min(mergeable, key=lambda p: self.merges[p])
            seq = self._apply_merge(seq, pair, self.merges[pair])
        return seq

In [5]:
PIECE_RE = re.compile(r"\w+|[^\w\s]")


class OffsetTokenizer:
    """BPE tokenizer that also tracks character offsets."""

    def __init__(self, bpe):
        self.bpe = bpe
        self.eos = bpe.special_to_id["</s>"]
        self.pad = bpe.special_to_id["<pad>"]
        self.cache = {}

    def _piece(self, piece):
        r = self.cache.get(piece)
        if r is None:
            ids = self.bpe.encode_piece(piece)
            lens = [len(self.bpe.id_to_bytes[i]) for i in ids]
            cs, ce, b = [], [], 0
            if piece.isascii():
                for L in lens:
                    cs.append(b); ce.append(b + L); b += L
            else:
                cb = [0]
                for ch in piece:
                    cb.append(cb[-1] + len(ch.encode("utf-8")))
                for L in lens:
                    cs.append(bisect_right(cb, b) - 1)
                    ce.append(bisect_left(cb, b + L))
                    b += L
            r = (ids, cs, ce)
            self.cache[piece] = r
        return r

    def encode(self, text):
        ids, st, en, real = [], [], [], []
        n = len(text)
        for m in PIECE_RE.finditer(text):
            pids, cs, ce = self._piece(m.group())
            base = m.start()
            for i, a, b in zip(pids, cs, ce):
                ids.append(i); st.append(base + a); en.append(base + b); real.append(True)
            e = m.end()
            if e >= n or text[e].isspace():
                ids.append(self.eos); st.append(e); en.append(e); real.append(False)
        return (np.array(ids, dtype=np.int64), np.array(st, dtype=np.int64),
                np.array(en, dtype=np.int64), np.array(real, dtype=bool))

In [6]:
def read_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)


def _get(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    raise KeyError(f"none of {keys} in {list(d)}")


def parse_entities(raw):
    """Accepts [s,e,label(,text)] lists or dicts with start/end/label(/text) style keys."""
    if isinstance(raw, dict):
        raw = _get(raw, ("entities", "annotations", "spans", "labels"))
    out = []
    for e in raw or []:
        if isinstance(e, (list, tuple)):
            s, en, l = e[0], e[1], e[2]
            t = e[3] if len(e) > 3 else None
        elif isinstance(e, dict):
            s = _get(e, ("start", "start_char", "start_offset", "begin", "s"))
            en = _get(e, ("end", "end_char", "end_offset", "stop", "e"))
            l = _get(e, ("label", "type", "entity", "tag", "l"))
            t = e.get("text", e.get("entity_text", e.get("span", None)))
        else:
            raise ValueError(f"unrecognised entity format: {e!r}")
        out.append((int(s), int(en), str(l), t))
    return out


def load_fold(text_path, gold_path=None):
    texts = read_json(text_path)
    texts = {str(k): (v["text"] if isinstance(v, dict) else v) for k, v in texts.items()}
    gold = None
    if gold_path is not None:
        g = read_json(gold_path)
        gold = {str(k): parse_entities(v) for k, v in g.items()}
        missing = set(texts) - set(gold)
        for k in missing:
            gold[k] = []
    return texts, gold


def check_offset_convention(texts, gold, n=2000):
    """Verifies gold uses [start, end) offsets (text[s:e] == entity_text) when entity text is given."""
    ok_excl = ok_incl = tot = 0
    for did, ents in gold.items():
        t = texts[did]
        for s, e, l, txt in ents:
            if txt is None:
                continue
            tot += 1
            ok_excl += t[s:e] == txt
            ok_incl += t[s:e + 1] == txt
            if tot >= n:
                break
        if tot >= n:
            break
    return tot, ok_excl, ok_incl


# Tagging
def tag_ids(labels):
    """O=0, B-label_k = 1+2k, I-label_k = 2+2k."""
    return 1 + 2 * len(labels)


def resolve_overlaps(ents):
    """Flat BIO cannot represent nested/overlapping gold: keep leftmost-longest, count the rest."""
    ents = sorted(ents, key=lambda x: (x[0], -(x[1] - x[0])))
    kept, last_end, dropped = [], -1, 0
    for s, e, l, *_ in ents:
        if s >= last_end:
            kept.append((s, e, l)); last_end = e
        else:
            dropped += 1
    return kept, dropped


def tag_tokens(st, en, real, ents, label2id):
    """Per-token tag ids. Non-real tokens (</s>) get -100. A token belongs to an entity when it
    overlaps it by at least half of the token's own length."""
    T = len(st)
    tags = np.full(T, -100, dtype=np.int64)
    ridx = np.nonzero(real)[0]
    tags[ridx] = 0
    rs, re_ = st[ridx], en[ridx]
    for s, e, l in ents:
        k = label2id[l]
        j = int(np.searchsorted(re_, s, side="right"))
        first = True
        while j < len(ridx) and rs[j] < e:
            ov = min(int(re_[j]), e) - max(int(rs[j]), s)
            if ov > 0 and 2 * ov >= int(re_[j] - rs[j]):
                tags[ridx[j]] = (1 + 2 * k) if first else (2 + 2 * k)
                first = False
            j += 1
    return tags


def boundary_alignment(st, en, real, ents):
    """Fraction of gold entities whose start AND end coincide with token edges (ceiling for strict match)."""
    starts = set(st[real].tolist()); ends = set(en[real].tolist())
    ok = [(s in starts and e in ends) for s, e, l in ents]
    return ok


# Windows
def window_starts(n, W, stride):
    if n <= W:
        return [0]
    starts = list(range(0, n - W + 1, stride))
    if starts[-1] != n - W:
        starts.append(n - W)
    return starts


def build_train_windows(enc_docs, tags_docs, W, stride, pad_id, keep_empty=0.3, seed=0):
    """enc_docs: {id: (ids, st, en, real)}; tags_docs: {id: tags}. Returns (ids[N,W], labels[N,W])."""
    rng = random.Random(seed)
    X, Y = [], []
    n_all = n_kept = 0
    for did, (ids, st, en, real) in enc_docs.items():
        tg = tags_docs[did]
        for s in window_starts(len(ids), W, stride):
            wi, wl = ids[s:s + W], tg[s:s + W]
            n_all += 1
            has_ent = bool((wl > 0).any())
            if not has_ent and rng.random() > keep_empty:
                continue
            n_kept += 1
            if len(wi) < W:
                pad = W - len(wi)
                wi = np.concatenate([wi, np.full(pad, pad_id, dtype=np.int64)])
                wl = np.concatenate([wl, np.full(pad, -100, dtype=np.int64)])
            X.append(wi); Y.append(wl)
    print(f"windows: {n_kept}/{n_all} kept (keep_empty={keep_empty})")
    return torch.from_numpy(np.stack(X)), torch.from_numpy(np.stack(Y))

In [7]:
ROPE_BASE = 10000.0

# Architecture 1: Decoder-only
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 2: Encoder-decoder
class MultiHeadAttention(nn.Module):
    """Multi-head attention with optional causal masking and cross-attention."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

# Architecture 3: Hybrid Prefix-LM
class PrefixAwareAttention(nn.Module):
    """Prefix-LM attention: bidirectional within the prefix, causal afterwards."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            # Build the mask here so DataParallel doesn't scatter it along the batch dimension.
            flat_logits = logits.view(-1, logits.size(-1))
            flat_targets = targets.view(-1)
            pos = torch.arange(T, device=idx.device)
            mask = (pos + 1 >= prefix_len).to(flat_logits.dtype)
            mask = mask.unsqueeze(0).expand(B, -1).reshape(-1)
            per_tok = F.cross_entropy(flat_logits, flat_targets, reduction="none")
            loss = (per_tok * mask).sum() / mask.sum().clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 4: Lightweight (RoPE, RMSNorm, SwiGLU, GQA)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos/sin: (T, d_head/2)
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 5: RoPE + RMSNorm + SwiGLU + QK-Norm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class QKNormAttention(nn.Module):
    """Causal attention with RoPE and per-head QK RMSNorm."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [8]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")


def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg, sd = ck["config"], ck["state_dict"]
    g = cfg.get
    if arch == "decoder":
        m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec":
        m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"),
                             g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid":
        m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight":
        m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"),
                          g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm":
        m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else:
        raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    max_len = g("max_seq_len") or g("max_src_len")
    return m, cfg, max_len


def block_list_name(arch):
    return "encoder_blocks" if arch == "encdec" else "blocks"


def strip_unused(m, arch):
    """Drop the LM-only parts (output head, decoder stack) once the weights are loaded."""
    m.lm_head = nn.Identity()
    if arch == "encdec":
        m.decoder_blocks = nn.ModuleList()
        m.dec_ln_f = nn.Identity()


def prune_layers(m, arch, keep):
    """KD student: keep only the listed transformer blocks (initialised from Phase-1 weights)."""
    name = block_list_name(arch)
    blocks = getattr(m, name)
    setattr(m, name, nn.ModuleList([blocks[i] for i in keep]))


class NERModel(nn.Module):
    def __init__(self, arch, backbone, d_model, num_tags, bidirectional=True, head_dropout=0.1):
        super().__init__()
        self.arch, self.backbone, self.bidir = arch, backbone, bidirectional
        self.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(d_model, num_tags))
        if bidirectional and arch in CAUSAL_ARCHS:
            # Allow each token to see both left and right context for NER.
            for mod in backbone.modules():
                if hasattr(mod, "causal_mask"):
                    mod.causal_mask.fill_(True)

    def features(self, ids):
        m, T = self.backbone, ids.shape[1]
        if self.arch == "decoder":
            x = m.drop(m.token_emb(ids) + m.pos_emb[:T].unsqueeze(0))
            for b in m.blocks: x = b(x)
            return m.ln_f(x)
        if self.arch == "encdec":
            return m.encode(ids)
        if self.arch == "hybrid":
            x = m.drop(m.token_emb(ids) + m.pos_emb[:T].unsqueeze(0))
            plen = T if self.bidir else 0
            for b in m.blocks: x = b(x, plen)
            return m.ln_f(x)
        x = m.drop(m.token_emb(ids))
        for b in m.blocks: x = b(x, m.rope_cos, m.rope_sin)
        return m.norm_f(x)

    def forward(self, ids):
        return self.head(self.features(ids))

In [9]:

class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B A x          (additive low-rank update, Hu et al.)"""
    def __init__(self, base, r=16, alpha=32, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA+ applies a low-rank multiplicative update: W' = (I + (alpha/r) BA)W0."""
    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features))
        self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))

    def forward(self, x):
        h = F.linear(x, self.base.weight)                       # W0 x
        y = h + self.s * F.linear(F.linear(h, self.A), self.B)  # (I + s B A) W0 x
        return y + self.base.bias if self.base.bias is not None else y


class Adapter(nn.Module):
    """Houlsby bottleneck adapter: x + up(gelu(down(x))), up initialised to zero (identity at start)."""
    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up, self.act = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.GELU()
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class Adapted(nn.Module):
    def __init__(self, inner, d, bottleneck=64):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck)

    def forward(self, *a, **k):
        return self.adapter(self.inner(*a, **k))


def _replace_linears(root, targets, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(mod, cname, make(child)); n += 1
    return n


def apply_method(ner, method, r=16, alpha=32, targets=("q_proj", "v_proj"), bottleneck=64, unfreeze_last=2):
    """Freezes/unfreezes parameters and injects PEFT modules. The NER head is always trainable."""
    bb = ner.backbone
    if method == "full":
        for p in ner.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora":
            assert _replace_linears(bb, targets, lambda b: LoRALinear(b, r, alpha)) > 0
        elif method == "lorma":
            assert _replace_linears(bb, targets, lambda b: LoRMALinear(b, r, alpha)) > 0
        elif method == "adapter":
            d = ner.head[1].in_features
            for blk in getattr(bb, block_list_name(ner.arch)):
                for cname, child in list(blk.named_children()):
                    if cname in ("attn", "self_attn", "ff"):
                        setattr(blk, cname, Adapted(child, d, bottleneck))
        elif method == "bitfit":
            for n, p in bb.named_parameters():
                if n.endswith(".bias"): p.requires_grad = True
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                    for p in mod.parameters(): p.requires_grad = True
        elif method == "partial":
            blocks = getattr(bb, block_list_name(ner.arch))
            for blk in list(blocks)[-unfreeze_last:]:
                for p in blk.parameters(): p.requires_grad = True
            for n, p in bb.named_parameters():
                if n.split(".")[0] in ("ln_f", "norm_f", "enc_ln_f"): p.requires_grad = True
        else:
            raise ValueError(method)
    for p in ner.head.parameters(): p.requires_grad = True
    tr = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in ner.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [10]:
def build_transition_masks(num_labels):
    K = 1 + 2 * num_labels
    allowed = np.zeros((K, K), dtype=bool)
    for b in range(K):
        for a in range(K):
            if b == 0 or b % 2 == 1:
                allowed[a, b] = True
            else:  # I-x requires previous B-x or I-x
                k = (b - 2) // 2
                allowed[a, b] = a in (1 + 2 * k, 2 + 2 * k)
    start_ok = np.array([not (b > 0 and b % 2 == 0) for b in range(K)])
    return allowed, start_ok


def viterbi_tags(logp, allowed, start_ok):
    N, K = logp.shape
    NEG = -1e9
    trans = np.where(allowed, 0.0, NEG).astype(np.float32)
    dp = logp[0] + np.where(start_ok, 0.0, NEG)
    back = np.zeros((N, K), dtype=np.int16)
    for t in range(1, N):
        sc = dp[:, None] + trans
        back[t] = sc.argmax(0)
        dp = sc.max(0) + logp[t]
    tags = np.zeros(N, dtype=np.int64)
    tags[-1] = int(dp.argmax())
    for t in range(N - 1, 0, -1):
        tags[t - 1] = back[t, tags[t]]
    return tags


def tags_to_spans(tags, st, en, labels):
    """tags/st/en are over REAL tokens only. Returns [(start,end,label)], strict [start,end) char offsets."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        t = int(t)
        if t == 0:
            if cur: spans.append(cur); cur = None
        elif t % 2 == 1:
            if cur: spans.append(cur)
            cur = [int(st[i]), int(en[i]), labels[(t - 1) // 2], (t - 1) // 2]
        else:
            k = (t - 2) // 2
            if cur and cur[3] == k:
                cur[1] = int(en[i])
            else:
                if cur: spans.append(cur)
                cur = [int(st[i]), int(en[i]), labels[k], k]
    if cur: spans.append(cur)
    seen, out = set(), []
    for s, e, l, _ in spans:
        if e > s and (s, e, l) not in seen:      # evaluator rejects empty and duplicate spans
            seen.add((s, e, l)); out.append((s, e, l))
    return out


def strict_prf(truth, pred, labels):
    """Re-implementation of evaluate_phase2.py (macro over ALL labels; absent label -> 0)."""
    stats = {l: [0, 0, 0] for l in labels}
    for did in truth:
        g = set(truth[did]); p = set(pred.get(did, []))
        for l in labels:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    P, R, F1s, per = [], [], [], {}
    for l in labels:
        tp, fp, fn = stats[l]
        pr = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0
        P.append(pr); R.append(rc); F1s.append(f); per[l] = (pr, rc, f, tp + fn)
    return sum(F1s) / len(F1s), sum(P) / len(P), sum(R) / len(R), per

In [11]:
def build_ner(arch, phase1_path, method, num_tags, bidirectional, keep_every=None, peft_kwargs=None):
    bb, cfg, max_len = load_phase1_backbone(arch, phase1_path)
    strip_unused(bb, arch)
    if keep_every:
        n_blocks = len(getattr(bb, block_list_name(arch)))
        prune_layers(bb, arch, list(range(0, n_blocks, keep_every)))
    ner = NERModel(arch, bb, cfg["d_model"], num_tags, bidirectional)
    apply_method(ner, method, **(peft_kwargs or {}))
    return ner.to(DEVICE), cfg, max_len


# Inference
@torch.no_grad()
def doc_logprobs(model, ids, W, stride, pad_id, bs=16):
    """Sliding windows over a document; per-token log-probs averaged over windows (centre-weighted)."""
    model.eval()
    T = len(ids)
    starts = window_starts(T, W, stride)
    acc = wsum = None
    for i in range(0, len(starts), bs):
        chunk = starts[i:i + bs]
        batch, lens = [], []
        for s in chunk:
            w = ids[s:s + W]; lens.append(len(w))
            if len(w) < W:
                w = np.concatenate([w, np.full(W - len(w), pad_id, dtype=np.int64)])
            batch.append(w)
        x = torch.from_numpy(np.stack(batch)).to(DEVICE)
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lp = F.log_softmax(model(x).float(), -1).cpu().numpy()
        if acc is None:
            acc = np.zeros((T, lp.shape[-1]), np.float32); wsum = np.zeros(T, np.float32)
        for j, s in enumerate(chunk):
            L = lens[j]; c = max((L - 1) / 2, 1)
            wt = (1.0 - 0.9 * np.abs(np.arange(L) - (L - 1) / 2) / c).astype(np.float32)
            acc[s:s + L] += lp[j, :L] * wt[:, None]; wsum[s:s + L] += wt
    return acc / wsum[:, None]


def decode_doc(lp, st, en, real, labels, allowed, start_ok, mode="viterbi", o_cut=math.log(0.999)):
    r = np.nonzero(real)[0]
    lp, st, en = lp[r], st[r], en[r]
    if len(lp) == 0:
        return []
    if mode == "greedy":
        return tags_to_spans(lp.argmax(1), st, en, labels)
    tags = np.zeros(len(lp), dtype=np.int64)
    unsure = lp[:, 0] < o_cut                      # tokens that are certainly 'O' are fixed; Viterbi only elsewhere
    i, n = 0, len(lp)
    while i < n:
        if not unsure[i]:
            i += 1; continue
        j = i
        while j < n and unsure[j]:
            j += 1
        tags[i:j] = viterbi_tags(lp[i:j], allowed, start_ok)
        i = j
    return tags_to_spans(tags, st, en, labels)


def predict_docs(model, enc_docs, labels, W, stride, pad_id, mode="viterbi", bs=16):
    allowed, start_ok = build_transition_masks(len(labels))
    out = {}
    for n, (did, (ids, st, en, real)) in enumerate(enc_docs.items()):
        lp = doc_logprobs(model, ids, W, stride, pad_id, bs)
        out[did] = decode_doc(lp, st, en, real, labels, allowed, start_ok, mode)
    return out


# Training
def kd_loss(s_logits, t_logits, y, weight, T, alpha):
    m = y != -100
    s, t, yy = s_logits[m].float(), t_logits[m].float(), y[m]
    hard = F.cross_entropy(s, yy, weight=weight)
    soft = F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
    return alpha * hard + (1 - alpha) * soft


def fit(ner, Xtr, Ytr, evaluate_fn, ckpt_dir, epochs, bs, accum, lr, head_mult, wd, warmup_frac,
        class_weight, teacher=None, kd_T=2.0, kd_alpha=0.5, seed=0):
    os.makedirs(ckpt_dir, exist_ok=True)
    last_p, best_p = os.path.join(ckpt_dir, "last.pt"), os.path.join(ckpt_dir, "best.pt")
    model = nn.DataParallel(ner) if NUM_GPUS > 1 else ner
    head_ids = {id(p) for p in ner.head.parameters()}
    params = [p for p in ner.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([
        {"params": [p for p in params if id(p) not in head_ids], "lr": lr},
        {"params": [p for p in params if id(p) in head_ids], "lr": lr * head_mult}], weight_decay=wd)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=bs, shuffle=True, drop_last=True,
                        generator=torch.Generator().manual_seed(seed))
    spe = max(len(loader) // accum, 1)
    total, warm = epochs * spe, max(int(warmup_frac * epochs * spe), 1)
    base_lrs = [g["lr"] for g in opt.param_groups]
    cw = class_weight.to(DEVICE) if class_weight is not None else None

    start_ep, best, hist = 0, -1.0, []
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        ner.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start_ep, best, hist = ck["epoch"], ck["best"], ck["hist"]
        print(f"resumed from epoch {start_ep}/{epochs} (best dev macro-F1 so far {best:.4f})")

    step = start_ep * spe
    t0 = time.time()
    for ep in range(start_ep, epochs):
        model.train(); run, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, (x, y) in enumerate(loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                logits = model(x)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(x)
                    loss = kd_loss(logits.reshape(-1, logits.size(-1)), t_logits.reshape(-1, t_logits.size(-1)),
                                   y.reshape(-1), cw, kd_T, kd_alpha)
                else:
                    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), y.reshape(-1),
                                           weight=cw, ignore_index=-100)
                loss = loss / accum
            scaler.scale(loss).backward()
            run += loss.item() * accum; nb += 1
            if (i + 1) % accum == 0:
                f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
                for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            if nb % 100 == 0:
                print(f"  ep {ep+1} batch {nb}/{len(loader)} loss {run/nb:.4f}", flush=True)
        f1, p, r, per = evaluate_fn(ner)
        hist.append({"epoch": ep + 1, "train_loss": run / max(nb, 1), "dev_f1": f1, "dev_p": p, "dev_r": r})
        print(f"epoch {ep+1}/{epochs} | loss {run/max(nb,1):.4f} | dev strict macro F1 {f1:.4f} P {p:.4f} R {r:.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f1 > best:
            best = f1
            torch.save({"trainable": trainable_state(ner), "f1": f1}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        torch.save({"trainable": trainable_state(ner), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": ep + 1, "best": best, "hist": hist}, last_p + ".tmp")
        os.replace(last_p + ".tmp", last_p)
    if os.path.exists(best_p):
        ner.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [12]:
# Data
def get_path(rel):
    if LOCAL_DATA_DIR:
        return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P2_DIR}/{rel}", repo_type="dataset", token=hf_token)

labels = [l.strip() for l in open(get_path("Train/labels.txt"), encoding="utf-8") if l.strip()]
label2id = {l: i for i, l in enumerate(labels)}
NUM_TAGS = 1 + 2 * len(labels)
assert len(labels) == 12, labels

all_texts, all_gold, fold_of = {}, {}, {}
for fold in ("fold1", "fold2"):
    t, g = load_fold(get_path(f"Train/{fold}-text.json"), get_path(f"Train/{fold}-gold.json"))
    for k in t:
        assert k not in all_texts, f"doc id {k} appears in both folds - prefix ids before merging"
        all_texts[k], all_gold[k], fold_of[k] = t[k], g[k], fold
test_texts, _ = load_fold(get_path("Test/fold3-text.json"))
print(f"train docs {len(all_texts)} | test docs {len(test_texts)} | labels {labels}")

if SMOKE:
    keep_ids = sorted(all_texts)[:60]
    all_texts = {k: all_texts[k] for k in keep_ids}; all_gold = {k: all_gold[k] for k in keep_ids}
    fold_of = {k: fold_of[k] for k in keep_ids}
    test_texts = {k: test_texts[k] for k in sorted(test_texts)[:5]}
tot, ok_e, ok_i = check_offset_convention(all_texts, all_gold)
print(f"offset convention check on {tot} gold entities with text: [start,end) matches {ok_e}, [start,end] matches {ok_i}")
if tot and ok_e < 0.99 * tot:
    print("!! WARNING: gold offsets do not look like [start, end) - inspect before trusting any score !!")
for bad in [(d, e) for d, es in all_gold.items() for e in es if e[2] not in label2id][:3]:
    raise ValueError(f"gold label not in labels.txt: {bad}")

ids_all = sorted(all_texts)
rng = random.Random(SEED)
if DEV_MODE == "fold2":
    dev_ids = [d for d in ids_all if fold_of[d] == "fold2"]
else:
    shuf = ids_all[:]; rng.shuffle(shuf); dev_ids = shuf[:max(int(DEV_FRAC * len(shuf)), 5)]
dev_set = set(dev_ids)
train_ids = [d for d in ids_all if d not in dev_set]
print(f"train {len(train_ids)} docs | dev {len(dev_ids)} docs (mode={DEV_MODE})")

bpe = BPE.load(TOKENIZER_PATH)
tok = OffsetTokenizer(bpe)
t0 = time.time()
enc = {}
for n, d in enumerate(ids_all):
    enc[d] = tok.encode(all_texts[d])
    if (n + 1) % 100 == 0: print(f"  tokenised {n+1}/{len(ids_all)}", flush=True)
test_enc = {d: tok.encode(t) for d, t in test_texts.items()}
print(f"tokenised in {time.time()-t0:.0f}s | vocab {bpe.vocab_size}")

# Data checks
tags, n_drop, n_ent, ok_by_label, n_by_label = {}, 0, 0, {l: 0 for l in labels}, {l: 0 for l in labels}
tag_count = np.zeros(NUM_TAGS, dtype=np.int64)
train_set = set(train_ids)
for d in ids_all:
    ids, st, en, real = enc[d]
    ents_full = [(s, e, l) for s, e, l, _ in all_gold[d]]
    for (s, e, l), ok in zip(ents_full, boundary_alignment(st, en, real, ents_full)):
        n_by_label[l] += 1; ok_by_label[l] += int(ok)
    kept, dropped = resolve_overlaps(all_gold[d]); n_drop += dropped; n_ent += len(ents_full)
    tags[d] = tag_tokens(st, en, real, kept, label2id)
    if d in train_set: tag_count += np.bincount(tags[d][tags[d] >= 0], minlength=NUM_TAGS)
align = sum(ok_by_label.values()) / max(n_ent, 1)
print(f"gold entities {n_ent} | dropped as nested/overlapping (flat BIO): {n_drop} ({100*n_drop/max(n_ent,1):.2f}%)")
print(f"BOUNDARY-ALIGNMENT CEILING (gold edges that land on token edges): {100*align:.2f}%  <- max achievable recall")
for l in labels:
    print(f"   {l:10s} n={n_by_label[l]:6d} aligned={100*ok_by_label[l]/max(n_by_label[l],1):6.2f}%")
lens = np.array([len(enc[d][0]) for d in ids_all])
print(f"tokens/doc: median {int(np.median(lens))}, max {lens.max()} | docs shorter than window: {(lens < WINDOW).sum()}")

dev_truth = {d: [(s, e, l) for s, e, l, _ in all_gold[d]] for d in dev_ids}
dev_enc = {d: enc[d] for d in dev_ids}
pad_id = tok.pad

train docs 70 | test docs 35 | labels ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
offset convention check on 2000 gold entities with text: [start,end) matches 2000, [start,end] matches 2
train 53 docs | dev 7 docs (mode=random)
tokenised in 1s | vocab 12000
gold entities 6933 | dropped as nested/overlapping (flat BIO): 0 (0.00%)
BOUNDARY-ALIGNMENT CEILING (gold edges that land on token edges): 99.31%  <- max achievable recall
   APP        n=   321 aligned= 98.13%
   RESP       n=   234 aligned= 99.15%
   A.COUNSEL  n=   170 aligned=100.00%
   R.COUNSEL  n=   166 aligned=100.00%
   JUDGE      n=   220 aligned=100.00%
   WIT        n=   187 aligned=100.00%
   AUTH       n=   782 aligned= 98.98%
   COURT      n=   837 aligned= 97.97%
   STAT       n=   990 aligned= 98.99%
   PREC       n=  1004 aligned= 99.50%
   DATE       n=  1345 aligned=100.00%
   CASENO     n=   677 aligned=100.00%
tokens/doc: median 10457, max 45470 |

In [13]:
_win_cache = {}
_eval_checked = False
LAST_RUN_T0 = time.time()

def backup(path):
    if not HF_BACKUP_REPO or not os.path.exists(path):
        return
    from huggingface_hub import HfApi
    HfApi().upload_file(path_or_fileobj=path, path_in_repo=os.path.relpath(path, "/kaggle/working"),
                        repo_id=HF_BACKUP_REPO, repo_type="model", token=hf_token)

def predict_and_save(model, enc_docs, W, lp_path=None):
    allowed, start_ok = build_transition_masks(len(labels))
    preds, lps = {}, {}
    for did, (ids, st, en, real) in enc_docs.items():
        lp = doc_logprobs(model, ids, W, INFER_STRIDE, pad_id)
        preds[did] = decode_doc(lp, st, en, real, labels, allowed, start_ok, DECODE)
        if lp_path:
            lps[did] = np.maximum(lp, -60.0).astype(np.float16)
    if lp_path:
        np.savez(lp_path, **lps)
    return preds

def write_submission(preds, work, zip_path):
    pred_path = f"{work}/prediction.json"
    json.dump({d: [list(x) for x in v] for d, v in preds.items()}, open(pred_path, "w"))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

def official_check(work):
    global _eval_checked
    if _eval_checked or not EVAL_SCRIPT:
        return
    _eval_checked = True
    r = subprocess.run(["python", EVAL_SCRIPT, "--truth", f"{work}/dev_truth.json", "--prediction", f"{work}/dev_pred.json",
                        "--labels", f"{work}/labels.txt"], capture_output=True, text=True)
    print(r.stdout[-1500:] or "(no stdout)")
    if r.returncode != 0:
        print("evaluate_phase2 failed - check its CLI with:  !python", EVAL_SCRIPT, "-h\n", r.stderr[-800:])

def run_experiment(method, kd=False, bidirectional=BIDIRECTIONAL):
    run_id = f"{ARCH}_{method}{'_kd' if kd else ''}_{'bidir' if bidirectional else 'causal'}"
    work, sub_zip = f"{WORK_ROOT}/{run_id}", f"{SUB_DIR}/{run_id}.zip"
    if os.path.exists(sub_zip) and os.path.exists(f"{work}/meta.json"):
        print(f"[skip] {run_id} already finished")
        return json.load(open(f"{work}/meta.json"))
    os.makedirs(work, exist_ok=True)
    print(f"\n{'=' * 20} {run_id} {'=' * 20}")
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    ner, cfg, max_len = build_ner(ARCH, MODEL_PATH, method, NUM_TAGS, bidirectional,
                                  keep_every=STUDENT_KEEP_EVERY if kd else None, peft_kwargs=PEFT)
    W = min(WINDOW, max_len)
    n_trainable = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in ner.parameters())

    teacher = None
    if kd:
        teacher_ckpt = f"{WORK_ROOT}/{ARCH}_full_bidir/ckpt/best.pt"
        if not os.path.exists(teacher_ckpt):
            raise FileNotFoundError("KD needs the finished full fine-tune run first: " + teacher_ckpt)
        teacher, _, _ = build_ner(ARCH, MODEL_PATH, "full", NUM_TAGS, True, peft_kwargs=PEFT)
        teacher.load_state_dict(torch.load(teacher_ckpt, map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False
        print(f"teacher {sum(p.numel() for p in teacher.parameters())/1e6:.1f}M -> student {n_total/1e6:.1f}M params")

    if W not in _win_cache:
        _win_cache[W] = build_train_windows({d: enc[d] for d in train_ids}, {d: tags[d] for d in train_ids},
                                            W, TRAIN_STRIDE, pad_id, KEEP_EMPTY, SEED)
    Xtr, Ytr = _win_cache[W]
    cw = torch.ones(NUM_TAGS); cw[0] = O_WEIGHT

    def evaluate_dev(m):
        return strict_prf(dev_truth, predict_docs(m, dev_enc, labels, W, INFER_STRIDE, pad_id, DECODE), labels)

    t_start = time.time()
    hist, best = fit(ner, Xtr, Ytr, evaluate_dev, f"{work}/ckpt", EPOCHS, BATCH_SIZE, GRAD_ACCUM, DEFAULT_LR[method],
                     HEAD_LR_MULT, WEIGHT_DECAY, WARMUP_FRAC, cw, teacher, KD_T, KD_ALPHA, SEED)
    train_min = (time.time() - t_start) / 60

    dev_pred = predict_and_save(ner, dev_enc, W, f"{work}/lp_dev.npz" if SAVE_LP else None)
    f1, p, r, per = strict_prf(dev_truth, dev_pred, labels)
    print(f"BEST DEV strict macro F1 {f1:.4f} | P {p:.4f} | R {r:.4f}")
    for l in labels:
        print(f"   {l:10s} P {per[l][0]:.3f}  R {per[l][1]:.3f}  F1 {per[l][2]:.3f}  (support {per[l][3]})")
    json.dump({d: [list(x) for x in v] for d, v in dev_truth.items()}, open(f"{work}/dev_truth.json", "w"))
    json.dump({d: [list(x) for x in v] for d, v in dev_pred.items()}, open(f"{work}/dev_pred.json", "w"))
    open(f"{work}/labels.txt", "w").write("\n".join(labels) + "\n")
    official_check(work)

    test_pred = predict_and_save(ner, test_enc, W, f"{work}/lp_test.npz" if SAVE_LP else None)
    write_submission(test_pred, work, sub_zip)

    meta = {"run": run_id, "arch": ARCH, "method": method, "kd": kd, "bidirectional": bidirectional, "decode": DECODE,
            "trainable_M": round(n_trainable / 1e6, 3), "total_M": round(n_total / 1e6, 2), "lr": DEFAULT_LR[method],
            "epochs": EPOCHS, "window": W, "dev_macroF1": round(f1, 4), "dev_P": round(p, 4), "dev_R": round(r, 4),
            "train_min": round(train_min, 1), "test_entities": sum(len(v) for v in test_pred.values()),
            "per_label_f1": {l: round(per[l][2], 4) for l in labels}, "history": hist}
    json.dump(meta, open(f"{work}/meta.json", "w"), indent=1)
    last = f"{work}/ckpt/last.pt"
    if os.path.exists(last):
        os.remove(last)
    for pth in (sub_zip, f"{work}/ckpt/best.pt", f"{work}/meta.json"):
        backup(pth)
    print(f"submission: {sub_zip} | test entities {meta['test_entities']} | {train_min:.1f} min")
    del ner, teacher
    torch.cuda.empty_cache()
    return meta

results = []
run_list = RUNS[:1] if SMOKE else RUNS
for spec in run_list:
    if (time.time() - LAST_RUN_T0) / 3600 > TIME_BUDGET_H:
        print("time budget reached - stopping; rerun this cell to continue (finished runs are skipped)")
        break
    results.append(run_experiment(**spec))


==================== hybrid_full_bidir ====================
[full] trainable 108.424M / 108.42M (100.00%)
windows: 1601/1727 kept (keep_empty=0.3)
  ep 1 batch 100/100 loss 0.8585
epoch 1/1 | loss 0.8585 | dev strict macro F1 0.0317 P 0.0215 R 0.0600 | 0.8 min
BEST DEV strict macro F1 0.0317 | P 0.0215 | R 0.0600
   APP        P 0.000  R 0.000  F1 0.000  (support 13)
   RESP       P 0.000  R 0.000  F1 0.000  (support 24)
   A.COUNSEL  P 0.000  R 0.000  F1 0.000  (support 15)
   R.COUNSEL  P 0.000  R 0.000  F1 0.000  (support 19)
   JUDGE      P 0.000  R 0.000  F1 0.000  (support 28)
   WIT        P 0.000  R 0.000  F1 0.000  (support 22)
   AUTH       P 0.000  R 0.000  F1 0.000  (support 106)
   COURT      P 0.258  R 0.720  F1 0.380  (support 100)
   STAT       P 0.000  R 0.000  F1 0.000  (support 114)
   PREC       P 0.000  R 0.000  F1 0.000  (support 100)
   DATE       P 0.000  R 0.000  F1 0.000  (support 107)
   CASENO     P 0.000  R 0.000  F1 0.000  (support 68)
Examples: 7
APP: P=

In [14]:
import pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob(f"{WORK_ROOT}/*/meta.json"))]
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("history", "per_label_f1")} for r in rows])
df.sort_values("dev_macroF1", ascending=False).to_csv(f"{WORK_ROOT}/results.csv", index=False)
df.sort_values("dev_macroF1", ascending=False)

,run,arch,method,kd,bidirectional,decode,trainable_M,total_M,lr,epochs,window,dev_macroF1,dev_P,dev_R,train_min,test_entities
0,hybrid_full_bidir,hybrid,full,False,True,viterbi,108.424,108.42,0.0001,1,512,0.0317,0.0215,0.06,0.9,232


## Ensemble

In [15]:
ENSEMBLE_GLOBS = [f"{WORK_ROOT}/*", "/kaggle/input/**/cs781_phase2/*"]
ENSEMBLE_MEMBERS = None
ENSEMBLE_TOPK = 4
ENSEMBLE_WEIGHTED = True

cands = {}
for g in ENSEMBLE_GLOBS:
    for d in glob.glob(g, recursive=True):
        if all(os.path.exists(f"{d}/{n}") for n in ("lp_dev.npz", "lp_test.npz", "meta.json")):
            cands[os.path.basename(d)] = d
info = {k: json.load(open(f"{d}/meta.json")) for k, d in cands.items()}
for k in sorted(info, key=lambda k: -info[k]["dev_macroF1"]):
    print(f"{k:40s} dev F1 {info[k]['dev_macroF1']:.4f}")

members = ENSEMBLE_MEMBERS or sorted(info, key=lambda k: -info[k]["dev_macroF1"])[:ENSEMBLE_TOPK]
weights = np.array([info[k]["dev_macroF1"] for k in members] if ENSEMBLE_WEIGHTED else [1.0] * len(members))
weights = weights / weights.sum()
print("ensemble members:", dict(zip(members, weights.round(3))))

def ensemble_predict(enc_docs, which):
    zs = [np.load(f"{cands[k]}/lp_{which}.npz") for k in members]
    allowed, start_ok = build_transition_masks(len(labels))
    preds = {}
    for did, (ids, st, en, real) in enc_docs.items():
        arrs = [z[did].astype(np.float32) for z in zs]
        assert all(len(a) == len(ids) for a in arrs), f"token count mismatch on doc {did}: members use different tokenizers"
        prob = sum(w * np.exp(a) for w, a in zip(weights, arrs))
        preds[did] = decode_doc(np.log(np.maximum(prob, 1e-9)), st, en, real, labels, allowed, start_ok, DECODE)
    return preds

ens_dev = ensemble_predict(dev_enc, "dev")
f1, p, r, per = strict_prf(dev_truth, ens_dev, labels)
print(f"ENSEMBLE DEV strict macro F1 {f1:.4f} | P {p:.4f} | R {r:.4f}  (best single member: {max(info[k]['dev_macroF1'] for k in members):.4f})")

hybrid_full_bidir                        dev F1 0.0317
ensemble members: {'hybrid_full_bidir': np.float64(1.0)}
ENSEMBLE DEV strict macro F1 0.0317 | P 0.0215 | R 0.0600  (best single member: 0.0317)


In [16]:
ens_test = ensemble_predict(test_enc, "test")
ens_id = f"ensemble_{len(members)}models"
ens_work = f"{WORK_ROOT}/{ens_id}"
os.makedirs(ens_work, exist_ok=True)
write_submission(ens_test, ens_work, f"{SUB_DIR}/{ens_id}.zip")
print("saved", f"{SUB_DIR}/{ens_id}.zip", "| test entities", sum(len(v) for v in ens_test.values()))

saved /kaggle/working/submissions_smoke/ensemble_1models.zip | test entities 232


## Bundle

In [17]:
BUNDLE_TOP = 2
top = sorted(info, key=lambda k: -info[k]["dev_macroF1"])[:BUNDLE_TOP] if info else []
bundle = f"/kaggle/working/{ARCH}_phase2_lp_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_STORED) as z:
    for k in top:
        for n in ("lp_dev.npz", "lp_test.npz", "meta.json"):
            z.write(f"{cands[k]}/{n}", f"cs781_phase2/{k}/{n}")
print(f"{os.path.getsize(bundle)/1e6:.0f} MB  {bundle}")
print("submissions:", sorted(os.listdir(SUB_DIR)))

9 MB  /kaggle/working/hybrid_phase2_lp_bundle.zip
submissions: ['ensemble_1models.zip', 'hybrid_full_bidir.zip']
